# Day 1 â€” PySpark: DataFrame Filtering
### filter(), where(), isin(), between(), isNull(), HAVING equivalent
### Target: Data Engineer Interviews

> **Roadmap Day:** 1 Â· **Date:** Monday, June 15, 2026  
> **Prerequisite:** Run `00_prerequisites.ipynb` first â€” Java and PySpark must be installed.  
> **Same problems as SQL Day 1 â€” same dataset, PySpark API.**

---
## Setup â€” SparkSession

In [1]:
import os, sys

# Must be set BEFORE any pyspark import â€” Spark reads these at import time
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

print("Environment set. Run next cell to start Spark.")

Environment set. Run next cell to start Spark.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, avg, count, min as spark_min, max as spark_max,
    round as spark_round, year, to_date, lower
)
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

spark = SparkSession.builder \
    .appName("Day01_Filtering") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready.")

Spark 3.5.6 ready.


---
## 1. The employees DataFrame

Mirrors the `employees` table from `setup_tables.sql` exactly â€” same data, same columns.

In [3]:
# Same data as setup_tables.sql employees table
data = [
    (1,  "Alice Johnson",  "alice@company.com",   "Engineering",  95000.0, 8000.0,  None, "2019-03-15", "active"),
    (2,  "Bob Smith",      "bob@company.com",     "Engineering",  65000.0, 3000.0,  1,    "2021-07-01", "active"),
    (3,  "Carol White",    "carol@company.com",   "Finance",      85000.0, 7000.0,  None, "2018-01-10", "active"),
    (4,  "Dave Brown",     "dave@company.com",    "Finance",      60000.0, 2500.0,  3,    "2022-04-20", "active"),
    (5,  "Eve Davis",      "eve@company.com",     "HR",           75000.0, 5000.0,  None, "2017-06-05", "active"),
    (6,  "Frank Miller",   "frank@company.com",   "HR",           50000.0, 1500.0,  5,    "2023-01-15", "active"),
    (7,  "Grace Wilson",   "grace@company.com",   "Marketing",    80000.0, 6000.0,  None, "2020-09-01", "active"),
    (8,  "Henry Moore",    "henry@company.com",   "Marketing",    55000.0, 2000.0,  7,    "2022-11-10", "inactive"),
    (9,  "Iris Taylor",    "iris@company.com",    "Engineering", 110000.0,10000.0,  1,    "2016-02-28", "active"),
    (10, "Jack Anderson",  "jack@company.com",    "Operations",   72000.0, 4000.0,  None, "2019-08-12", "active"),
    (11, "Karen Thomas",   "karen@company.com",   "Operations",   48000.0, 1000.0,  10,   "2023-03-01", "active"),
    (12, "Leo Jackson",    "leo@company.com",     "Engineering",  78000.0, 5500.0,  1,    "2020-05-17", "active"),
    (13, "Mia Harris",     "mia@company.com",     "Finance",      70000.0, 4500.0,  3,    "2021-02-28", "active"),
    (14, "Nathan Clark",   "nathan@company.com",  "Marketing",    45000.0,  800.0,  7,    "2023-06-01", "active"),
    (15, "Olivia Lewis",   "olivia@company.com",  "HR",           52000.0, 1800.0,  5,    "2022-08-15", "active"),
]

schema = StructType([
    StructField("id",         IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("email",      StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("salary",     FloatType(),   True),
    StructField("bonus",      FloatType(),   True),
    StructField("manager_id", IntegerType(), True),
    StructField("hire_date",  StringType(),  True),
    StructField("status",     StringType(),  True),
])

df = spark.createDataFrame(data, schema=schema)

# Cast hire_date to DateType for proper date operations
df = df.withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd"))

df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: float (nullable = true)
 |-- bonus: float (nullable = true)
 |-- manager_id: integer (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- status: string (nullable = true)



In [4]:
df.select("id", "name", "department", "salary", "hire_date", "status").show(truncate=False)

+---+-------------+-----------+--------+----------+--------+
|id |name         |department |salary  |hire_date |status  |
+---+-------------+-----------+--------+----------+--------+
|1  |Alice Johnson|Engineering|95000.0 |2019-03-15|active  |
|2  |Bob Smith    |Engineering|65000.0 |2021-07-01|active  |
|3  |Carol White  |Finance    |85000.0 |2018-01-10|active  |
|4  |Dave Brown   |Finance    |60000.0 |2022-04-20|active  |
|5  |Eve Davis    |HR         |75000.0 |2017-06-05|active  |
|6  |Frank Miller |HR         |50000.0 |2023-01-15|active  |
|7  |Grace Wilson |Marketing  |80000.0 |2020-09-01|active  |
|8  |Henry Moore  |Marketing  |55000.0 |2022-11-10|inactive|
|9  |Iris Taylor  |Engineering|110000.0|2016-02-28|active  |
|10 |Jack Anderson|Operations |72000.0 |2019-08-12|active  |
|11 |Karen Thomas |Operations |48000.0 |2023-03-01|active  |
|12 |Leo Jackson  |Engineering|78000.0 |2020-05-17|active  |
|13 |Mia Harris   |Finance    |70000.0 |2021-02-28|active  |
|14 |Nathan Clark |Marke

---
## 2. filter() and where() â€” Two Names, One Function

`filter()` and `where()` are **identical aliases** â€” same behavior, interchangeable.  
Both accept a Column expression or a SQL string.

In [6]:
# Column expression â€” recommended for production code
df.filter(col("department") == "Engineering") \
  .select("id", "name", "department", "salary") \
  .orderBy(col("salary").desc()) \
  .show()

Py4JJavaError: An error occurred while calling o72.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 6 in stage 1.0 failed 1 times, most recent failure: Lost task 6.0 in stage 1.0 (TID 7) (SF-LAP-197 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "C:\Users\hariom\Downloads\python_sql_pyspark\myenv\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 1100, in main
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 11) than that in driver 3.13, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.py

In [ ]:
# where() â€” same result
df.where("department = 'Engineering'") \
  .select("id", "name", "department", "salary") \
  .orderBy(col("salary").desc()) \
  .show()

In [ ]:
# Three equivalent syntaxes â€” all return the same rows
r1 = df.filter(col("department") == "Engineering")
r2 = df.filter(df["department"] == "Engineering")
r3 = df.filter("department = 'Engineering'")

print("All three produce the same result:",
      r1.count() == r2.count() == r3.count() == True)

---
## 3. between() â€” Inclusive Range Filter

Equivalent to SQL `BETWEEN low AND high` â€” inclusive on both ends.

In [ ]:
# Numeric range â€” salary 60000â€“100000 inclusive
df.filter(col("salary").between(60000, 100000)) \
  .select("id", "name", "department", "salary") \
  .orderBy("salary") \
  .show()

In [ ]:
# Date range â€” hired in 2022
df.filter(col("hire_date").between("2022-01-01", "2022-12-31")) \
  .select("id", "name", "hire_date") \
  .orderBy("hire_date") \
  .show()

In [ ]:
# Alternative: year() function for year-based filtering
df.filter(year(col("hire_date")) == 2022) \
  .select("id", "name", "hire_date") \
  .show()

---
## 4. isin() â€” Match Any Value in a List

In [ ]:
# isin â€” equivalent to SQL IN (...)
df.filter(col("department").isin("Engineering", "Finance")) \
  .select("id", "name", "department") \
  .orderBy("department", "name") \
  .show()

In [ ]:
# isin with a Python list variable
target_depts = ["Engineering", "Finance"]

df.filter(col("department").isin(target_depts)) \
  .select("name", "department") \
  .show()

In [ ]:
# NOT IN â€” use ~ (tilde) to negate
df.filter(~col("department").isin("HR", "Marketing")) \
  .select("name", "department") \
  .orderBy("department", "name") \
  .show()

---
## 5. String Pattern Matching

PySpark has dedicated string methods that mirror SQL `LIKE` and go beyond it.

In [ ]:
# startswith â€” SQL LIKE 'A%'
df.filter(col("name").startswith("A")) \
  .select("id", "name") \
  .show()

In [ ]:
# endswith â€” SQL LIKE '%son'
df.filter(col("name").endswith("son")) \
  .select("name") \
  .show()

In [ ]:
# contains â€” SQL LIKE '%ar%'
df.filter(col("name").contains("ar")) \
  .select("name") \
  .show()

In [ ]:
# like() â€” same SQL % wildcard syntax
df.filter(col("name").like("A%")) \
  .select("name") \
  .show()

In [ ]:
# rlike() â€” full regex (much more powerful than SQL LIKE)
# Match names starting with 'A' or 'I'
df.filter(col("name").rlike("^[AI]")) \
  .select("name") \
  .show()

In [ ]:
# Case-insensitive â€” use lower()
# SQL ILIKE 'a%' equivalent
df.filter(lower(col("name")).startswith("a")) \
  .select("name") \
  .show()

In [ ]:
# Email domain filter â€” gmail users
df.filter(col("email").endswith("@company.com")) \
  .select("name", "email") \
  .show()

---
## 6. isNull() and isNotNull()

In [ ]:
# isNull â€” employees with no manager (top-level)
df.filter(col("manager_id").isNull()) \
  .select("id", "name", "department", "manager_id") \
  .orderBy("department") \
  .show()

In [ ]:
# isNotNull â€” employees who report to someone
df.filter(col("manager_id").isNotNull()) \
  .select("id", "name", "department", "manager_id") \
  .orderBy("manager_id", "name") \
  .show()

In [ ]:
# Gotcha: comparing to None does NOT match null rows
df_test = spark.createDataFrame([(1, None), (2, 100.0)], ["id", "val"])

# Wrong â€” returns 0 rows
wrong_count = df_test.filter(col("val") == None).count()
print(f"col == None: {wrong_count} rows (wrong approach)")

# Correct â€” returns 1 row
right_count = df_test.filter(col("val").isNull()).count()
print(f"col.isNull(): {right_count} row (correct approach)")

---
## 7. Combining Conditions â€” &, |, ~

> **Critical:** Every condition must be in `()` parentheses when combining with `&` or `|`.  
> Python's operator precedence causes silent, hard-to-debug bugs without them.

In [ ]:
# AND â€” use & with parentheses around each condition
df.filter(
    (col("department") == "Engineering") &
    (col("salary").between(60000, 100000))
).select("id", "name", "department", "salary") \
 .orderBy(col("salary").desc()) \
 .show()

In [ ]:
# OR â€” use |
df.filter(
    col("name").startswith("A") |
    (year(col("hire_date")) == 2022)
).select("id", "name", "department", "hire_date") \
 .orderBy("hire_date") \
 .show()

In [ ]:
# NOT â€” use ~
df.filter(~col("department").isin("HR", "Marketing")) \
  .select("name", "department") \
  .orderBy("department", "name") \
  .show()

In [ ]:
# Three conditions chained
df.filter(
    (col("status") == "active") &
    (col("salary") > 70000) &
    (col("department").isin("Engineering", "Finance"))
).select("name", "department", "salary", "status") \
 .orderBy(col("salary").desc()) \
 .show()

In [ ]:
# Also: use & | ~ NOT Python 'and' 'or' 'not'
try:
    df.filter(col("department") == "Engineering" and col("salary") > 80000).show()
except Exception as e:
    print(f"Using 'and' fails: {type(e).__name__}: {e}")
    print("Use & instead of 'and', | instead of 'or', ~ instead of 'not'")

In [ ]:
# SQL: GROUP BY department HAVING AVG(salary) > 70000

df.groupBy("department") \
  .agg(
      spark_round(avg("salary"), 2).alias("avg_salary"),
      count("*").alias("headcount")
  ) \
  .filter(col("avg_salary") > 70000) \
  .orderBy(col("avg_salary").desc()) \
  .show()

In [ ]:
# All departments â€” comparison
df.groupBy("department") \
  .agg(
      spark_round(avg("salary"), 0).alias("avg_salary"),
      count("*").alias("headcount"),
      spark_min("salary").alias("min_sal"),
      spark_max("salary").alias("max_sal")
  ) \
  .orderBy(col("avg_salary").desc()) \
  .show()

In [ ]:
# WHERE + HAVING equivalent
# Active employees (row filter) â†’ departments with 2+ people and avg > 60000 (group filter)
df.filter(col("status") == "active") \
  .groupBy("department") \
  .agg(
      count("*").alias("headcount"),
      spark_round(avg("salary"), 0).alias("avg_salary")
  ) \
  .filter(
      (col("headcount") >= 2) &
      (col("avg_salary") > 60000)
  ) \
  .orderBy(col("avg_salary").desc()) \
  .show()

---
## 9. Day 1 Problems â€” Official Solutions

Try writing your solution before running each cell.

In [ ]:
# Problem 1 (Easy)
# Find all employees in the 'Engineering' department with salary between 60000 and 100000

print("Problem 1 â€” Engineering dept, salary 60kâ€“100k")
print("=" * 52)

df.filter(
    (col("department") == "Engineering") &
    (col("salary").between(60000, 100000))
).select("id", "name", "department", "salary") \
 .orderBy(col("salary").desc()) \
 .show()

# Expected: Alice 95000, Leo 78000, Bob 65000
# Iris 110000 â†’ excluded (above 100000)

In [ ]:
# Problem 2 (Easy)
# Find departments where the average salary is greater than 70000

print("Problem 2 â€” Departments with avg salary > 70000")
print("=" * 52)

df.groupBy("department") \
  .agg(
      spark_round(avg("salary"), 2).alias("avg_salary"),
      count("*").alias("headcount")
  ) \
  .filter(col("avg_salary") > 70000) \
  .orderBy(col("avg_salary").desc()) \
  .show()

In [ ]:
# Problem 3 (Medium)
# Find employees whose name starts with 'A' OR hired in 2022

print("Problem 3 â€” Name starts with A OR hired in 2022")
print("=" * 52)

df.filter(
    col("name").startswith("A") |
    (year(col("hire_date")) == 2022)
).select("id", "name", "department", "salary", "hire_date") \
 .orderBy("hire_date") \
 .show()

---
## 10. SparkSQL Alternative â€” Same Queries via SQL String

In [ ]:
# Register as temp view â€” then use SQL directly
df.createOrReplaceTempView("employees")

# Problem 1 via SparkSQL
spark.sql("""
    SELECT id, name, department, salary
    FROM employees
    WHERE department = 'Engineering'
      AND salary BETWEEN 60000 AND 100000
    ORDER BY salary DESC
""").show()

In [ ]:
# Problem 2 via SparkSQL
spark.sql("""
    SELECT department,
           ROUND(AVG(salary), 2) AS avg_salary,
           COUNT(*) AS headcount
    FROM employees
    GROUP BY department
    HAVING AVG(salary) > 70000
    ORDER BY avg_salary DESC
""").show()

In [ ]:
# Problem 3 via SparkSQL â€” YEAR() function works in Spark SQL
spark.sql("""
    SELECT id, name, department, salary, hire_date
    FROM employees
    WHERE name LIKE 'A%'
       OR YEAR(hire_date) = 2022
    ORDER BY hire_date
""").show()

---
## 11. PySpark vs SQL â€” Full Mapping Table

| SQL | PySpark |
|-----|---------|
| `WHERE dept = 'Engineering'` | `filter(col("dept") == "Engineering")` |
| `WHERE salary BETWEEN 60000 AND 100000` | `filter(col("salary").between(60000, 100000))` |
| `WHERE dept IN ('Eng', 'Fin')` | `filter(col("dept").isin("Eng", "Fin"))` |
| `WHERE dept NOT IN ('HR')` | `filter(~col("dept").isin("HR"))` |
| `WHERE name LIKE 'A%'` | `filter(col("name").startswith("A"))` |
| `WHERE name ILIKE 'a%'` | `filter(lower(col("name")).startswith("a"))` |
| `WHERE salary IS NULL` | `filter(col("salary").isNull())` |
| `WHERE salary IS NOT NULL` | `filter(col("salary").isNotNull())` |
| `WHERE a AND b` | `filter((col_a) & (col_b))` |
| `WHERE a OR b` | `filter((col_a) \| (col_b))` |
| `WHERE NOT a` | `filter(~col_a)` |
| `HAVING AVG(sal) > 70000` | `.groupBy().agg(avg().alias("x")).filter(col("x") > 70000)` |

In [ ]:
spark.stop()
print("SparkSession stopped.")